
# Notebook 17 · GenAI para voz del cliente (VOC)

## Pregunta de negocio
¿Cómo convertir tickets, reviews y feedback abierto en insights accionables para marketing y CX?

## Qué aprenderás
- Cómo unificar VOC estructurada y no estructurada.
- Cómo detectar temas, fricciones y drivers de compra.
- Cómo preparar *prompt packs* para IA generativa con guardrails.

## Outputs
- `nb17_voc_issue_matrix.csv`
- `nb17_voc_prompt_pack.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

feedback = tables["feedback"].copy()
tickets = tables["tickets"].copy()
guardrails = tables["guardrails"].copy()

voc_feedback = feedback[[
    "feedback_id", "fecha_feedback_dt", "customer_id", "canal_feedback", "tipo_feedback",
    "sentimiento", "tema_principal", "texto_feedback"
]].rename(columns={
    "feedback_id": "registro_id",
    "fecha_feedback_dt": "fecha_evento",
    "canal_feedback": "canal",
    "texto_feedback": "texto",
})
voc_feedback["fuente"] = "feedback"

voc_tickets = tickets[[
    "service_ticket_id", "fecha_apertura_dt", "customer_id", "canal_soporte", "tipo_caso",
    "sentimiento_inicial", "motivo_contacto", "texto_cliente"
]].rename(columns={
    "service_ticket_id": "registro_id",
    "fecha_apertura_dt": "fecha_evento",
    "canal_soporte": "canal",
    "tipo_caso": "tipo_feedback",
    "sentimiento_inicial": "sentimiento",
    "motivo_contacto": "tema_principal",
    "texto_cliente": "texto",
})
voc_tickets["fuente"] = "tickets"

voc = pd.concat([voc_feedback, voc_tickets], ignore_index=True)
voc = voc.dropna(subset=["texto"]).copy()

voc.shape, voc.head()


((1410, 9),
   registro_id        fecha_evento customer_id             canal    tipo_feedback sentimiento    tema_principal                                              texto    fuente
 0   FB-000013 2024-09-01 16:24:45       C0015  email_postcompra  review_producto     Neutral  calidad_producto  En general bien, pero el acabado del Aplique B...  feedback
 1   FB-000301                 NaT       C0055        web_review  review_producto    Positivo  calidad_producto  Buen producto, materiales correctos y resultad...  feedback
 2   FB-000185                 NaT       C0016  email_postcompra  review_producto    Negativo  calidad_producto  El Sofá Zen 120cm no transmite la calidad que ...  feedback
 3   FB-000056                 NaT       C0027  email_postcompra  review_producto    Positivo  calidad_producto  Me ha sorprendido para bien la calidad del Coj...  feedback
 4   FB-000220                 NaT       C0067        web_review  review_producto     Neutral    medidas_ajuste  El Cama Nó

In [3]:

issue_matrix = (
    voc.groupby(["fuente", "tema_principal", "sentimiento"])
       .agg(
           registros=("registro_id", "count"),
           clientes=("customer_id", "nunique"),
       )
       .reset_index()
       .sort_values(["registros", "clientes"], ascending=False)
)

top_negative_themes = (
    issue_matrix[issue_matrix["sentimiento"].astype(str).str.lower().isin(["negativo", "negative"])]
    .groupby("tema_principal")["registros"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .reset_index()
    .rename(columns={"registros": "negative_mentions"})
)

top_terms = mpu.top_terms(voc["texto"], n=25)
issue_matrix.head(20), top_terms.head(15)


(      fuente      tema_principal sentimiento  registros  clientes
 11  feedback    precio_promocion    Positivo        117        69
 5   feedback    entrega_servicio    Positivo         88        60
 8   feedback      medidas_ajuste    Positivo         88        53
 10  feedback    precio_promocion     Neutral         83        54
 2   feedback    calidad_producto    Positivo         70        48
 14  feedback  tienda_experiencia    Positivo         63        49
 9   feedback    precio_promocion    Negativo         60        48
 0   feedback    calidad_producto    Negativo         55        40
 4   feedback    entrega_servicio     Neutral         50        38
 1   feedback    calidad_producto     Neutral         49        37
 7   feedback      medidas_ajuste     Neutral         43        38
 63   tickets             entrega     Neutral         43        35
 6   feedback      medidas_ajuste    Negativo         36        30
 62   tickets             entrega    Negativo         32      

In [4]:

prompt_rows = []
high_severity_rules = guardrails.sort_values(["severidad", "categoria_regla"])

for theme in top_negative_themes["tema_principal"].tolist():
    subset = voc[voc["tema_principal"] == theme].copy()
    sample_texts = subset["texto"].dropna().astype(str).head(5).tolist()
    channels = ", ".join(sorted(subset["canal"].dropna().astype(str).unique())[:5])
    sentiments = subset["sentimiento"].astype(str).value_counts(normalize=True).round(3).to_dict()

    prompt = f'''
Actúa como analista de cliente y marketing de precisión de FraSoHome.
Tema crítico detectado: {theme}.
Canales donde aparece: {channels}.
Distribución de sentimiento aproximada: {sentiments}.
Ejemplos de voz del cliente:
- {'; - '.join(sample_texts)}

Quiero que devuelvas:
1) insight principal,
2) causas probables,
3) impacto en conversión/retención,
4) recomendación de contenido, UX o servicio,
5) KPI para validar la mejora.
'''.strip()

    guardrail_text = " | ".join(high_severity_rules["descripcion_regla"].head(5).tolist())

    prompt_rows.append({
        "tema_principal": theme,
        "mentions_negativas": int(top_negative_themes.loc[top_negative_themes["tema_principal"] == theme, "negative_mentions"].iloc[0]),
        "sample_channels": channels,
        "guardrails_relevantes": guardrail_text,
        "prompt_pack": prompt,
        "brief_determinista": f"Priorizar revisión del tema '{theme}' porque concentra fricción repetida y afecta a experiencia/consideración. Validar mejora con reducción de menciones negativas y mejora de NPS/CSAT."
    })

prompt_pack = pd.DataFrame(prompt_rows).sort_values("mentions_negativas", ascending=False)
prompt_pack


,tema_principal,mentions_negativas,sample_channels,guardrails_relevantes,prompt_pack,brief_determinista
0,precio_promocion,60,"app_review, email_postcompra, encuesta_tienda,...","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'precio_promocion'...
1,calidad_producto,55,"app_review, email_postcompra, encuesta_tienda,...","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'calidad_producto'...
2,medidas_ajuste,36,"app_review, email_postcompra, encuesta_tienda,...","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'medidas_ajuste' p...
3,entrega,32,"Chat, Email, Teléfono, Tienda, WhatsApp","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'entrega' porque c...
4,entrega_servicio,28,"encuesta_tienda, nps_app, nps_email, post_devo...","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'entrega_servicio'...
5,info producto,27,"Chat, Email, Teléfono, Tienda, WhatsApp","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'info producto' po...
6,cupon,24,"Chat, Email, Teléfono, Tienda, WhatsApp","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'cupon' porque con...
7,stock tienda,21,"Chat, Email, Teléfono, Tienda, WhatsApp","Si hay promoción, especificar condiciones de f...",Actúa como analista de cliente y marketing de ...,Priorizar revisión del tema 'stock tienda' por...



## Lectura de negocio

Aquí la IA generativa no sustituye al análisis. Lo potencia.

El flujo recomendado es:

1. usar analítica para ordenar volumen, temas y fricción,
2. preparar *packs* de contexto con guardrails,
3. pedir a la GenAI una síntesis útil para decisiones de contenido, CX y CRM,
4. validar después en KPI reales.


In [5]:

path_matrix = mpu.export_csv(issue_matrix, "nb17_voc_issue_matrix.csv")
path_prompts = mpu.export_csv(prompt_pack, "nb17_voc_prompt_pack.csv")

print("Guardado:", path_matrix)
print("Guardado:", path_prompts)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb17_voc_issue_matrix.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb17_voc_prompt_pack.csv
